# Decision Trees

Basics (Daniel?):

- What is a tree?
- Root, Decision Node and Leaf
- Small Example
- Pros and Cons compared to Linear Regression etc.

Concepts (Clemens?):

- Different Algorithms (CART, ID3 etc.)
- Entropy
- Gini impurity
- Information Gain
- Pruning


**Calculate Gini Impurity [(source)](https://en.wikipedia.org/wiki/Decision_tree_learning#Gini_impurity):**
J... Labels
p... Probability of randomly selecting label
$$Gini = 1 - \sum_{i=1}^{J} p^2_i$$


In [146]:
import pandas as pd

In [147]:
# df = pd.DataFrame(
#     [
#         ["Green", 3, "Apple"],
#         ["Yellow", 3, "Apple"],
#         ["Red", 1, "Grape"],
#         ["Red", 1, "Grape"],
#         ["Yellow", 3, "Lemon"],
#     ],
#     columns=["color", "diameter", "fruit"],
# )

X_train = pd.DataFrame(
    [
        ["Green", 3],
        ["Yellow", 3],
        ["Red", 1],
        ["Red", 1],
        ["Yellow", 3],
    ],
    columns=["color", "diameter"],
)

y_train = pd.DataFrame(["Apple", "Apple", "Grape", "Grape", "Lemon"])

In [148]:
class Decision:
    def __init__(self, df, column_index, value):
        self.column_index = column_index
        self.column_name = df.columns[column_index]
        self.value = value

    def match(self, example):
        val = example[self.column_name]

        if isinstance(self.value, (int, float)):  # Numeric
            return val >= self.value
        return val == self.value  # Other

    def __repr__(self):
        condition = ">=" if isinstance(self.value, (int, float)) else "=="
        return f"{self.column_name} {condition} {self.value}?"

In [149]:
q = Decision(X_train, 0, "Yellow")  # Is column 0 (color) Yellow?

print('["Green", 3, "Apple"]: ', q.match(X_train.iloc[0]))
print('["Yellow", 3, "Apple"]: ', q.match(X_train.iloc[1]))

["Green", 3, "Apple"]:  False
["Yellow", 3, "Apple"]:  True


In [150]:
def partition(X, y, question):
    mask = X.apply(lambda row: question.match(row), axis=1)
    return X[mask], y[mask], X[~mask], y[~mask]


# Usage:
X_t, _, _, _ = partition(X_train, y_train, q)
print("Decision: ", q)
X_t

Decision:  color == Yellow?


,color,diameter
1,Yellow,3
4,Yellow,3


In [151]:
def gini(y):
    # automatically calculates probablities for each label
    probs = y.value_counts(normalize=True)
    return 1 - (probs**2).sum()


# 0% Chance of misclassification
print('Gini for ["Apple", "Apple"]:', gini(pd.Series(["Apple", "Apple"])))

# 50% Chance of misclassification
print('Gini for ["Apple", "Grape"]:', gini(pd.Series(["Apple", "Grape"])))

# 66.7% Chance of misclassification a random example from the dataset
print(
    'Gini for ["Apple", "Grape", "Peach"]:',
    gini(pd.Series(["Apple", "Grape", "Peach"])),
)

Gini for ["Apple", "Apple"]: 0.0
Gini for ["Apple", "Grape"]: 0.5
Gini for ["Apple", "Grape", "Peach"]: 0.6666666666666667


In [152]:
def information_gain(y_left, y_right, current_uncertainty):
    p = len(y_left) / (len(y_left) + len(y_right))
    return current_uncertainty - p * gini(y_left) - (1 - p) * gini(y_right)

In [153]:
current_uncertainty = gini(y_train)

X_t_green, y_t_green, X_f_green, y_f_green = partition(
    X_train, y_train, Decision(X_train, 0, "Green")
)

ig_green = information_gain(
    y_left=y_t_green,
    y_right=y_f_green,
    current_uncertainty=current_uncertainty,
)

X_t_red, y_t_red, X_f_red, y_f_red = partition(
    X_train, y_train, Decision(X_train, 0, "Red")
)

ig_red = information_gain(
    y_left=y_t_red, y_right=y_f_red, current_uncertainty=current_uncertainty
)

print(f"Information Gain for Green: {round(ig_green, 3)}")
print(f"Information Gain for Red: {round(ig_red, 3)}")

Information Gain for Green: 0.14
Information Gain for Red: 0.373


In [154]:
def find_best_split(X, y):
    best_gain = 0
    best_question = None
    current_uncertainty = gini(y)  # ... with respect to the label

    # Iterate over features
    for col_idx in range(len(X.columns)):
        # Try split for all values in column (e.g. Red, Yellow...)
        values = X.iloc[:, col_idx].unique()

        for val in values:
            question = Decision(X, col_idx, val)
            _, y_t, _, y_f = partition(X, y, question)

            if len(y_t) == 0 or len(y_f) == 0:
                continue

            # information gain from this split, with respect to the label column (e.g fruit)
            gain = information_gain(y_t, y_f, current_uncertainty)

            # Keep Track of best gain so far
            if gain >= best_gain:
                best_gain, best_question = gain, question

    return best_gain, best_question

In [155]:
find_best_split(X_train, y_train)

(np.float64(0.37333333333333324), diameter == 1?)

In [156]:
class Leaf:
    def __init__(self, y):
        self.predictions = y.value_counts().to_dict()


class Decision_Node:
    def __init__(self, question, true_branch, false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

In [157]:
def build_tree(X, y):
    gain, question = find_best_split(X, y)

    # No more decision to be performed, dead-end
    if gain == 0:
        return Leaf(y)

    # Ask best question
    X_t, y_t, X_f, y_f = partition(X, y, question)

    # Recursively building Tree
    true_branch = build_tree(X_t, y_t)
    false_branch = build_tree(X_f, y_f)

    return Decision_Node(question, true_branch=true_branch, false_branch=false_branch)

In [158]:
def print_tree(node, spacing=""):
    # Base case: we've reached a leaf
    if isinstance(node, Leaf):
        total = sum(node.predictions.values())
        # Calculate percentages for more intuitive reading
        summary = {
            k: f"{round((v / total) * 100, 1)}%" for k, v in node.predictions.items()
        }
        print(spacing + "└── Prediction: " + str(summary))
        return

    # Print the question at the current node
    print(spacing + "Query: " + str(node.question))

    # Handle the True branch
    print(spacing + "├── True:")
    print_tree(node.true_branch, spacing + "│   ")

    # Handle the False branch
    print(spacing + "└── False:")
    print_tree(node.false_branch, spacing + "    ")

In [159]:
# Not really interesting yet
tree = build_tree(X_train, y_train)
print_tree(tree)

Query: diameter == 1?
├── True:
│   └── Prediction: {('Grape',): '100.0%'}
└── False:
    Query: color == Yellow?
    ├── True:
    │   └── Prediction: {('Apple',): '50.0%', ('Lemon',): '50.0%'}
    └── False:
        └── Prediction: {('Apple',): '100.0%'}


In [160]:
def classify(row, node):
    # base-case, leaf can return its predictions
    if isinstance(node, Leaf):
        return node.predictions

    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)

In [161]:
test_df = pd.DataFrame(
    [
        ["Red", 1],  # Easy example: Grape
        ["Yellow", 3],  # Could either be Apple or Lemon
        ["Green", 5],  # How does it handle new Values - probably Apple?
        ["Blabla", -2],  # Faulty data
    ],
    columns=["color", "diameter"],
)

# Grape
print(classify(test_df.iloc[0], tree))

# Returns both Apple and lemon => we need to decide ourselves (e.g. random)
print(classify(test_df.iloc[1], tree))

# Apple
print(classify(test_df.iloc[2], tree))

# Whatever this is
print(classify(test_df.iloc[3], tree))

{('Grape',): 2}
{('Apple',): 1, ('Lemon',): 1}
{('Apple',): 1}
{('Apple',): 1}


Sources:

- [Video](https://www.youtube.com/watch?v=LDRbO9a6XPU)
- [Wikipedia](https://en.wikipedia.org/wiki/Decision_tree_learning#Gini_impurity)
- [Decision tree - geeksforgeeks](https://www.geeksforgeeks.org/machine-learning/decision-tree-implementation-python/)
- [Sklearn](https://scikit-learn.org/stable/modules/tree.html)
